# Практическая работа № 2. Разведочный анализ данных (EDA)
**Уровень B — полный EDA.** В репозитории доступен `tips.csv`, но отдельный «грязный» файл от преподавателя не приложен. Чтобы показать диагностику дефектов, ниже создаётся воспроизводимая копия с несколькими искусственно внесёнными пропусками и аномальными значениями; эти изменения явно отделены от исходных данных.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
RANDOM_STATE = 42
DATA_PATH = Path("../data/tips.csv") if Path("../data/tips.csv").exists() else Path("data/tips.csv")
original = pd.read_csv(DATA_PATH)
df = original.copy()
# Только учебная демонстрация: добавляем контролируемые дефекты в копию.
df.loc[[5, 19, 40], "total_bill"] = np.nan
df.loc[[12, 77], "day"] = np.nan
df.loc[10, "total_bill"] = df["total_bill"].median() * 8
print("Исходный размер:", original.shape, "| учебная копия:", df.shape)
df.head()

In [ ]:
def classify_column(s):
    if pd.api.types.is_numeric_dtype(s):
        return "числовой"
    return "категориальный"
feature_types = pd.DataFrame({"тип": df.dtypes.astype(str), "класс признака": [classify_column(df[c]) for c in df.columns]})
display(feature_types)
print("\nПропуски:")
display(df.isna().sum().rename("число пропусков").to_frame())
display(df.describe(include="all").T)

In [ ]:
numeric = df.select_dtypes(include=np.number).columns.tolist()
categorical = df.select_dtypes(exclude=np.number).columns.tolist()
fig, axes = plt.subplots(len(numeric), 2, figsize=(11, 4*len(numeric)))
for i, col in enumerate(numeric):
    axes[i, 0].hist(df[col].dropna(), bins=20, edgecolor="black")
    axes[i, 0].set(title=f"Распределение: {col}", xlabel=col, ylabel="Частота")
    sns.boxplot(x=df[col], ax=axes[i, 1])
    axes[i, 1].set(title=f"Boxplot: {col}", xlabel=col)
plt.tight_layout(); plt.show()

In [ ]:
for col in categorical:
    print(f"Частоты для {col}:")
    display(df[col].value_counts(dropna=False).to_frame("count"))
    df[col].fillna("Неизвестно").value_counts().plot(kind="bar", title=f"Категории: {col}")
    plt.xlabel(col); plt.ylabel("Количество"); plt.tight_layout(); plt.show()

In [ ]:
corr = df[numeric].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Корреляционная матрица числовых признаков"); plt.tight_layout(); plt.show()
# Наиболее коррелирующие пары по абсолютной корреляции
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(key=np.abs, ascending=False)
print("Наиболее коррелирующие пары:")
display(pairs.head(5).rename("correlation").to_frame())
if len(numeric) >= 2:
    sns.pairplot(df[numeric].dropna().sample(min(150, df[numeric].dropna().shape[0]), random_state=RANDOM_STATE))
    plt.show()

## Отчёт: обнаруженные проблемы и план обработки
1. В учебной копии есть пропуски в `total_bill` и `day`; для числового поля предлагается медианная импутация, для категориального — отдельная категория «Неизвестно».
2. В `total_bill` искусственно внесено аномально большое значение; проверить его по IQR и исходной записи, не удалять автоматически без предметного обоснования.
3. Числовые признаки имеют разные масштабы; для моделей, чувствительных к масштабу, применить стандартизацию внутри Pipeline.
4. Категориальные признаки требуют кодирования; редкие категории следует просмотреть по таблицам частот до возможного объединения.
5. Корреляция показывает линейную связь, но сама по себе не доказывает причинность.
6. `tips.csv` не является датасетом классификации с исходной целевой меткой, поэтому оценка дисбаланса классов здесь не применяется.
7. Искусственные дефекты нужны только для демонстрации; при получении файла преподавателя заменить учебную копию реальными данными.